In [ ]:
# Import necessary libraries
import numpy as np
import nibabel as nib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from dipy.io.image import load_nifti
from dipy.core.gradients import gradient_table

In [ ]:
# Load the preprocessed DWI data
dwi_path = 'derivatives/preprocessed_dwi/sub-01/sub-01_preprocessed_dwi.nii.gz'
bval_path = 'derivatives/preprocessed_dwi/sub-01/sub-01_preprocessed_dwi.bval'
bvec_path = 'derivatives/preprocessed_dwi/sub-01/sub-01_preprocessed_dwi.bvec'

# Load the data
dwi_data, affine = load_nifti(dwi_path)
bvals = np.loadtxt(bval_path)
bvecs = np.loadtxt(bvec_path).T
gtab = gradient_table(bvals, bvecs)

# Print data information
print(f"DWI data shape: {dwi_data.shape}")
print(f"Number of b-values: {len(bvals)}")
print(f"Number of gradient directions: {bvecs.shape[1]}")
print(f"Unique b-values: {np.unique(bvals)}")

# Visualize a slice of the DWI data
plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.imshow(dwi_data[:, :, dwi_data.shape[2]//2, 0], cmap='gray')
plt.title('First b=0 volume')
plt.subplot(122)
plt.imshow(dwi_data[:, :, dwi_data.shape[2]//2, -1], cmap='gray')
plt.title('Last diffusion volume')
plt.show()

In [ ]:
# Prepare data for machine learning
# Reshape the data to have samples (voxels) as rows and features (diffusion directions) as columns
n_samples = dwi_data.shape[0] * dwi_data.shape[1] * dwi_data.shape[2]
n_features = dwi_data.shape[3]

X = dwi_data.reshape(n_samples, n_features)

# For demonstration, let's create a binary classification task
# We'll classify voxels based on their mean intensity
mean_intensity = np.mean(X, axis=1)
threshold = np.median(mean_intensity)
y = (mean_intensity > threshold).astype(int)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

In [ ]:
# Train and evaluate Logistic Regression
logreg = LogisticRegression(max_iter=1000, n_jobs=-1)
logreg.fit(X_train_scaled, y_train)
logreg_score = logreg.score(X_test_scaled, y_test)
print(f"Logistic Regression Accuracy: {logreg_score:.4f}")

In [ ]:
# Train and evaluate XGBoost
xgb_model = xgb.XGBClassifier(n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)
xgb_score = xgb_model.score(X_test_scaled, y_test)
print(f"XGBoost Accuracy: {xgb_score:.4f}")

In [ ]:
# Create a simple neural network for diffusion modeling
class DiffusionModel(nn.Module):
    def __init__(self, input_dim):
        super(DiffusionModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc4(x))
        return x

# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1)

# Create the model
model = DiffusionModel(n_features)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
n_epochs = 20
batch_size = 256
train_losses = []
test_losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for i in range(0, len(X_train_tensor), batch_size):
        batch_X = X_train_tensor[i:i+batch_size]
        batch_y = y_train_tensor[i:i+batch_size]
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / (len(X_train_tensor) // batch_size))
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())
        predictions = (test_outputs > 0.5).float()
        accuracy = (predictions == y_test_tensor).float().mean()
        
    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}, Accuracy: {accuracy.item():.4f}")

# Plot training and test losses
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Test Losses')
plt.legend()
plt.show()